In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             mean_absolute_error, r2_score)

print("✅ Imports done.")

✅ Imports done.


In [2]:
#DataSet Builder

def _qubit_cost(algo, key):
    mapping = {
        ("RSA",   512):   2.0,  ("RSA",  1024):  4.0,
        ("RSA",  2048):   8.0,  ("RSA",  4096): 16.0,
        ("ECC",   160):   0.8,  ("ECC",   256):  1.2,
        ("ECC",   384):   1.8,  ("ECC",   521):  2.5,
        ("ECDSA", 256):   1.2,  ("ECDSA", 384):  1.8,
        ("DH",   2048):   8.0,  ("DH",   4096): 16.0,
        ("AES",   128): 999.0,  ("AES",   256): 999.0,
        ("ML-KEM",768):   0.0,
    }
    return mapping.get((algo, key), 8.0)

def _pq_security(algo, key):
    if algo in ("RSA", "ECC", "ECDSA", "DH"):
        return 0
    if algo == "AES":
        return key // 2
    if algo == "ML-KEM":
        return key
    return 0

def _risk_to_label(score):
    if score >= 70:  return "CRITICAL"
    if score >= 50:  return "HIGH"
    if score >= 30:  return "MEDIUM"
    return "LOW"

def _qday_year(algo, key):
    if algo == "ML-KEM":        return 9999
    if algo in ("RSA", "DH"):
        return {512: 2027, 1024: 2028, 2048: 2030, 4096: 2033}.get(key, 2030)
    if algo in ("ECC", "ECDSA"):
        return {160: 2026, 256: 2027, 384: 2028, 521: 2029}.get(key, 2027)
    if algo == "AES":
        return 9999 if key >= 256 else 2035
    return 2030

def build_dataset():
    rng = np.random.default_rng(42)
    templates = [
        ("AIS",              "ECDSA",  256,  9,  9,  2),
        ("ECDIS",            "RSA",   2048,  8, 10,  2),
        ("SATCOM",           "RSA",   2048,  9,  8,  3),
        ("Port_OT",          "RSA",   2048,  7, 10,  2),
        ("GMDSS",            "RSA",   1024,  9,  9,  1),
        ("Vessel_Email",     "RSA",   2048,  8,  6,  4),
        ("AIS_Firmware",     "RSA",   2048,  7,  8,  3),
        ("Engine_OT",        "AES",    128,  6,  7,  5),
        ("Bridge_Auth",      "AES",    256,  5,  8,  6),
        ("Cargo_Manifest",   "ECC",    256,  6,  7,  5),
        ("VSAT",             "ECC",    256,  7,  6,  4),
        ("Biometric_Access", "RSA",   2048,  4,  5,  7),
        ("PQC_Hybrid_Pilot", "ML-KEM", 768,  1,  2,  9),
        ("GNSS_Receiver",    "AES",    128,  6,  9,  5),
        ("Port_VPN",         "DH",    2048,  7,  9,  3),
        ("Cargo_Database",   "AES",    256,  5,  7,  6),
        ("Crew_PII",         "RSA",   2048,  9, 10,  2),
        ("Financial_Tx",     "RSA",   2048,  8, 10,  3),
        ("Chart_Updates",    "ECC",    384,  8,  9,  3),
        ("Legacy_AIS",       "RSA",   1024,  9,  9,  1),
    ]
    rows = []
    for (sys_name, algo, key, L, I, R) in templates:
        for _ in range(6):
            l = int(np.clip(L + rng.integers(-1, 2), 1, 10))
            i = int(np.clip(I + rng.integers(-1, 2), 1, 10))
            r = int(np.clip(R + rng.integers(-1, 2), 1, 10))
            algo_penalty = {"RSA": 38,"ECC": 32,"ECDSA": 32,"DH": 34,
                            "AES": 8,"ML-KEM": -35,"ML-DSA": -30}.get(algo, 20)
            readiness_boost = max(0, (5 - r) * 2.5)
            risk = float(np.clip(
                (l*0.8)+(i*0.9)-(r*0.5)+algo_penalty+readiness_boost+rng.normal(0,3.5),
                0, 100
            ))
            rows.append({
                "system": sys_name, "algorithm": algo, "key_size": key,
                "likelihood": l, "impact": i, "pqc_readiness": r,
                "qubits_m": _qubit_cost(algo, key),
                "pq_sec_bits": _pq_security(algo, key),
                "risk_score": round(risk, 2),
                "qday_year": _qday_year(algo, key),
                "label": _risk_to_label(risk),
            })
    return pd.DataFrame(rows)

df = build_dataset()
print(f"✅ Dataset built — {df.shape[0]} records | Labels: {df['label'].value_counts().to_dict()}")

✅ Dataset built — 120 records | Labels: {'HIGH': 62, 'LOW': 30, 'MEDIUM': 28}


In [3]:
#Train Model

def encode_features(df):
    enc = {}
    df2 = df.copy()
    for col in ("system", "algorithm"):
        le = LabelEncoder()
        df2[col + "_enc"] = le.fit_transform(df2[col])
        enc[col] = le
    feature_cols = ["algorithm_enc","key_size","likelihood","impact",
                    "pqc_readiness","qubits_m","pq_sec_bits"]
    return df2, feature_cols, enc

def train_models(df):
    df2, feat_cols, enc = encode_features(df)
    X       = df2[feat_cols].values
    y_cls   = df2["label"].values
    y_reg   = df2["risk_score"].values
    le_lbl  = LabelEncoder()
    y_enc   = le_lbl.fit_transform(y_cls)
    enc["label"] = le_lbl

    X_tr, X_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
        X, y_enc, y_reg, test_size=0.25, random_state=42, stratify=y_enc
    )
    rf_clf = RandomForestClassifier(n_estimators=300, max_depth=10,
                min_samples_leaf=2, class_weight="balanced",
                random_state=42, n_jobs=-1)
    rf_clf.fit(X_tr, yc_tr)

    rf_reg = RandomForestRegressor(n_estimators=300, max_depth=10,
                min_samples_leaf=2, random_state=42, n_jobs=-1)
    rf_reg.fit(X_tr, yr_tr)

    yc_pred = rf_clf.predict(X_te)
    yr_pred = rf_reg.predict(X_te)
    cv      = cross_val_score(rf_clf, X, y_enc, cv=5, scoring="accuracy")

    metrics = {
        "clf_report":  classification_report(yc_te, yc_pred,
                           target_names=le_lbl.classes_, output_dict=True),
        "confusion":   confusion_matrix(yc_te, yc_pred),
        "cv_mean":     cv.mean(), "cv_std": cv.std(),
        "reg_mae":     mean_absolute_error(yr_te, yr_pred),
        "reg_r2":      r2_score(yr_te, yr_pred),
        "feat_imp":    dict(zip(feat_cols, rf_clf.feature_importances_)),
        "label_names": le_lbl.classes_,
    }
    return rf_clf, rf_reg, enc, feat_cols, metrics

rf_clf, rf_reg, enc, feat_cols, metrics = train_models(df)
rpt = metrics["clf_report"]
print("✅ Models trained!")
print(f"   CV Accuracy  : {metrics['cv_mean']*100:.2f}% (±{metrics['cv_std']*100:.2f}%)")
print(f"   F1-Score (w) : {rpt['weighted avg']['f1-score']*100:.2f}%")
print(f"   Regressor R² : {metrics['reg_r2']:.4f}  |  MAE: {metrics['reg_mae']:.2f}")

✅ Models trained!
   CV Accuracy  : 86.67% (±12.75%)
   F1-Score (w) : 92.99%
   Regressor R² : 0.9532  |  MAE: 3.58


In [5]:
#Prediction Interface

def _breach_status(algo, key, qubit_year):
    qcost = _qubit_cost(algo, key)
    if qcost == 0:   return "IMMUNE — Post-Quantum Algorithm"
    if qcost >= 999: return "GROVER THREAT — upgrade to AES-256 recommended"
    avail = {2026:1.0,2027:2.0,2028:5.0,2029:7.0,
             2030:10.0,2031:12.0,2032:15.0,2033:20.0}
    q_avail = avail.get(qubit_year, qubit_year * 0.7)
    if q_avail >= qcost:
        return f"⚠ BREACH FEASIBLE in {qubit_year} ({q_avail:.0f}M qubits ≥ {qcost:.0f}M needed)"
    return f"✅ NOT FEASIBLE in {qubit_year} ({q_avail:.0f}M < {qcost:.0f}M qubits needed)"

def _recommend(algo, key, score, label):
    recs = []
    if algo in ("RSA","ECC","ECDSA","DH"):
        recs.append("🔴 MIGRATE  — Replace with ML-KEM (FIPS 203) for key exchange")
        recs.append("🔴 SIGN     — Use ML-DSA (FIPS 204) for digital signatures")
    if algo == "AES" and key < 256:
        recs.append("🟠 UPGRADE  — Increase AES key 128→256 (Grover resistance)")
    if algo == "ML-KEM":
        recs.append("🟢 SECURE   — Lattice-based algorithm, quantum safe")
    if label in ("CRITICAL","HIGH"):
        recs.append("🔴 PRIORITY — Schedule immediate crypto-audit")
        recs.append("🟠 INTERIM  — Deploy hybrid RSA + PQC wrapper now")
    if label == "MEDIUM":
        recs.append("🟡 PLAN     — Include in 12-month PQC migration cycle")
    if label == "LOW":
        recs.append("🟢 MONITOR  — Maintain crypto-agile architecture, review yearly")
    recs.append("📋 STANDARD — NIST FIPS 203/204 · IMO MSC-FAL.1/Circ.3")
    return recs

def predict(system, algorithm, key_size, likelihood, impact, pqc_readiness, qubit_year):
    algo = algorithm.upper().replace(" ","-")
    qubits_m = _qubit_cost(algo, key_size)
    pq_bits  = _pq_security(algo, key_size)
    qday     = _qday_year(algo, key_size)
    le_algo  = enc["algorithm"]
    if algo in le_algo.classes_:
        algo_enc = int(le_algo.transform([algo])[0])
    else:
        aliases = {"AES-256":"AES","AES-128":"AES","KYBER":"ML-KEM","DILITHIUM":"ML-DSA"}
        mapped   = aliases.get(algo,"RSA")
        algo_enc = int(le_algo.transform([mapped])[0]) if mapped in le_algo.classes_ else 0
    X = np.array([[algo_enc, key_size, likelihood, impact, pqc_readiness, qubits_m, pq_bits]])
    lbl_enc   = rf_clf.predict(X)[0]
    lbl_prob  = rf_clf.predict_proba(X)[0]
    label     = enc["label"].inverse_transform([lbl_enc])[0]
    risk      = float(np.clip(rf_reg.predict(X)[0], 0, 100))
    return {
        "system": system, "algorithm": algo, "key_size": key_size,
        "likelihood": likelihood, "impact": impact, "pqc_readiness": pqc_readiness,
        "qubit_year": qubit_year, "qubits_needed_m": qubits_m, "pq_sec_bits": pq_bits,
        "risk_score": round(risk, 2), "label": label,
        "label_proba": {enc["label"].classes_[i]: round(p*100,1) for i,p in enumerate(lbl_prob)},
        "qday_year": qday, "years_to_qday": max(0,qday-2026) if qday<9999 else None,
        "breach_status": _breach_status(algo, key_size, qubit_year),
        "recommendation": _recommend(algo, key_size, risk, label),
    }

print("✅ Prediction engine ready.")

✅ Prediction engine ready.


In [ ]:
#Prediction UI For The Maritime Sector

# ── Key size options per algorithm ────────────────────────────────────────────
KEY_OPTIONS = {
    "RSA":    [512, 1024, 2048, 4096],
    "ECC":    [160, 256, 384, 521],
    "ECDSA":  [256, 384],
    "DH":     [2048, 4096],
    "AES":    [128, 256],
    "ML-KEM": [768],
}

SYSTEMS = ["AIS","ECDIS","SATCOM","Port_OT","GMDSS","Vessel_Email",
           "AIS_Firmware","Engine_OT","Bridge_Auth","Cargo_Manifest",
           "VSAT","Biometric_Access","PQC_Hybrid_Pilot","GNSS_Receiver",
           "Port_VPN","Cargo_Database","Crew_PII","Financial_Tx",
           "Chart_Updates","Legacy_AIS","Custom"]

# ── Widgets ───────────────────────────────────────────────────────────────────
w_system  = widgets.Dropdown(options=SYSTEMS, value="AIS",
                description="System:", style={"description_width":"120px"},
                layout=widgets.Layout(width="340px"))

w_algo    = widgets.Dropdown(options=list(KEY_OPTIONS.keys()), value="RSA",
                description="Algorithm:", style={"description_width":"120px"},
                layout=widgets.Layout(width="340px"))

w_key     = widgets.Dropdown(options=KEY_OPTIONS["RSA"], value=2048,
                description="Key Size:", style={"description_width":"120px"},
                layout=widgets.Layout(width="340px"))

w_like    = widgets.IntSlider(value=8, min=1, max=10, step=1,
                description="Likelihood:", style={"description_width":"120px"},
                layout=widgets.Layout(width="420px"))

w_impact  = widgets.IntSlider(value=9, min=1, max=10, step=1,
                description="Impact:", style={"description_width":"120px"},
                layout=widgets.Layout(width="420px"))

w_ready   = widgets.IntSlider(value=2, min=1, max=10, step=1,
                description="PQC Readiness:", style={"description_width":"120px"},
                layout=widgets.Layout(width="420px"))

w_qyear   = widgets.IntSlider(value=2030, min=2026, max=2033, step=1,
                description="Qubit Year:", style={"description_width":"120px"},
                layout=widgets.Layout(width="420px"))

w_btn     = widgets.Button(description="▶  Run Assessment",
                button_style="primary",
                layout=widgets.Layout(width="200px", height="38px"))

out       = widgets.Output()

# ── Auto-update key size choices when algorithm changes ───────────────────────
def _on_algo_change(change):
    w_key.options = KEY_OPTIONS[change["new"]]
    w_key.value   = KEY_OPTIONS[change["new"]][0]

w_algo.observe(_on_algo_change, names="value")

# ── Run on button click ───────────────────────────────────────────────────────
def _on_run(b):
    with out:
        clear_output(wait=True)

        r = predict(
            system       = w_system.value,
            algorithm    = w_algo.value,
            key_size     = w_key.value,
            likelihood   = w_like.value,
            impact       = w_impact.value,
            pqc_readiness= w_ready.value,
            qubit_year   = w_qyear.value,
        )

        # ── Result summary ────────────────────────────────────────────────
        lvl_colour = {"CRITICAL":"#d62728","HIGH":"#ff7f0e","MEDIUM":"#f9c74f","LOW":"#2ca02c"}
        col = lvl_colour.get(r["label"], "blue")
        print("=" * 58)
        print(f"  VULNERABILITY : \033[1m{r['label']}\033[0m")
        print(f"  RISK SCORE    : {r['risk_score']:.1f} / 100")
        print(f"\n  Probability Breakdown:")
        for lbl, pct in sorted(r["label_proba"].items(), key=lambda x:-x[1]):
            bar = "█" * int(pct / 3)
            print(f"    {lbl:10s}  {bar:<33s} {pct:.1f}%")
        print(f"\n  Algorithm  : {r['algorithm']}-{r['key_size']}")
        sec_status = "BROKEN" if r["pq_sec_bits"]==0 else ("WEAKENED" if r["pq_sec_bits"]<112 else "SAFE")
        print(f"  Post-Q Sec : {r['pq_sec_bits']} bits  [{sec_status}]")
        qday_disp = "IMMUNE (PQC)" if r["qday_year"]==9999 else str(r["qday_year"])
        print(f"  Q-Day Year : {qday_disp}")
        if r["years_to_qday"] is not None:
            print(f"  Time Left  : ~{r['years_to_qday']} year(s) from 2026")
        print(f"\n  Breach Status:")
        print(f"    {r['breach_status']}")
        print(f"\n  Recommendations:")
        for rec in r["recommendation"]:
            print(f"    {rec}")
        print("=" * 58)

        # ── Chart 1: Feature Importance ───────────────────────────────────
        names_map = {
            "algorithm_enc":"Algorithm",     "key_size":"Key Size",
            "likelihood":"Likelihood",        "impact":"Impact",
            "pqc_readiness":"PQC Readiness",  "qubits_m":"Qubits Needed (M)",
            "pq_sec_bits":"Post-Q Sec Bits",
        }
        sorted_f = sorted(metrics["feat_imp"].items(), key=lambda x: x[1], reverse=True)
        names  = [names_map.get(k,k) for k,_ in sorted_f]
        values = [v for _,v in sorted_f]

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.barh(names, values, height=0.55)
        ax.set_xlabel("Feature Importance (Gini)")
        ax.set_title("Random Forest — Feature Importance")
        ax.set_xlim(0, max(values) * 1.25)
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.4)
        for idx, val in enumerate(values):
            ax.text(val + 0.003, idx, f"{val:.3f}", va="center", fontsize=9)
        fig.text(0.99, 0.01,
                 f"CV Accuracy: {metrics['cv_mean']*100:.1f}% (±{metrics['cv_std']*100:.1f}%)",
                 ha="right", fontsize=8)
        plt.tight_layout()
        plt.show()

        # ── Chart 2: Confusion Matrix ─────────────────────────────────────
        cm     = metrics["confusion"]
        lbls   = metrics["label_names"]
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(cm, cmap="Blues", aspect="auto")
        ax.set_xticks(range(len(lbls))); ax.set_xticklabels(lbls, rotation=30, ha="right")
        ax.set_yticks(range(len(lbls))); ax.set_yticklabels(lbls)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title("Confusion Matrix — Vulnerability Classification")
        thresh = cm.max() / 2
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i,j]), ha="center", va="center",
                        fontsize=12, fontweight="bold",
                        color="white" if cm[i,j] > thresh else "black")
        plt.colorbar(im, ax=ax)
        acc = metrics["clf_report"]["accuracy"]
        fig.text(0.5,0.01,f"Accuracy: {acc*100:.1f}%", ha="center", fontsize=9)
        plt.tight_layout()
        plt.show()

        # ── Chart 3: Q-Day Timeline ───────────────────────────────────────
        systems_qday = [
            ("Legacy AIS (RSA-1024)",   2028),
            ("GMDSS (RSA-1024)",        2028),
            ("AIS/ECDIS (ECC-256)",     2027),
            ("SATCOM (RSA-2048)",       2030),
            ("Port OT VPN (DH-2048)",   2030),
            ("Vessel Email (RSA-2048)", 2030),
            ("AES-256 Systems",         2035),
            ("PQC Hybrid (ML-KEM)",     9999),
        ]
        yd    = [y if y < 9999 else 2038 for _, y in systems_qday]
        sys_l = [s for s, _ in systems_qday]

        fig, ax = plt.subplots(figsize=(11, 5))
        ax.barh(sys_l, [y - 2026 for y in yd], left=2026, height=0.55)
        ax.axvline(2026, linewidth=2, linestyle="--", label="Now (2026)")
        ax.axvline(r["qubit_year"], linewidth=1.5, linestyle=":",
                   label=f"Your Qubit Year ({r['qubit_year']})", color="red")
        ax.set_xlabel("Year")
        ax.set_title("Q-Day Timeline: When Does Each Algorithm Get Broken?")
        ax.set_xlim(2025, 2040)
        ax.grid(axis="x", alpha=0.4)
        ax.invert_yaxis()
        for bar, (name, yr) in zip(ax.patches, systems_qday):
            lx  = (2038 if yr >= 9999 else yr) + 0.1
            txt = "SAFE (PQC)" if yr >= 9999 else str(yr)
            ax.text(lx, bar.get_y() + bar.get_height()/2, txt, va="center", fontsize=8.5)
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

w_btn.on_click(_on_run)

# ── Layout & display ──────────────────────────────────────────────────────────
header = widgets.HTML("<h3 style='margin-bottom:8px'>🔐 Quantum Threat Assessment — Maritime Infrastructure</h3>")
col1   = widgets.VBox([w_system, w_algo, w_key])
col2   = widgets.VBox([w_like, w_impact, w_ready, w_qyear])
row    = widgets.HBox([col1, widgets.HTML("&nbsp;&nbsp;&nbsp;"), col2])

display(widgets.VBox([header, row, w_btn, out])) 